In [ ]:
# Static xT Pipeline

StatsBomb → SPADL → Karun xT → Player Leaderboard

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

from socceraction.data.statsbomb import StatsBombLoader
import socceraction.spadl as spadl

PROJECT_ROOT = Path.cwd().parent

OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/tstanton/Desktop/soccer-event-data/soccer-xT-project')

In [2]:
SBL = StatsBombLoader(
    getter="remote",
    creds=None
)

games = SBL.games(
    competition_id=43,
    season_id=3
)

games.head()

/opt/homebrew/Caskroom/miniforge/base/envs/soccerxt/lib/python3.11/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee
0,7585,3,43,Round of 16,4,2018-07-03 20:00:00,769,768,1,1,Otkritie Bank Arena,Mark Geiger
1,7570,3,43,Group Stage,3,2018-06-28 20:00:00,768,782,0,1,Stadion Kaliningrad,Damir Skomina
2,7586,3,43,Round of 16,4,2018-07-03 16:00:00,790,773,1,0,Saint-Petersburg Stadium,Damir Skomina
3,7557,3,43,Group Stage,3,2018-06-25 20:00:00,797,780,1,1,Mordovia Arena,Enrique Cáceres
4,7542,3,43,Group Stage,2,2018-06-20 14:00:00,780,788,1,0,Stadion Luzhniki,Mark Geiger


In [3]:
game_id = games.iloc[0]["game_id"]

events = SBL.events(game_id)

events.head()

/opt/homebrew/Caskroom/miniforge/base/envs/soccerxt/lib/python3.11/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,game_id,event_id,period_id,team_id,player_id,type_id,type_name,index,timestamp,minute,...,team_name,duration,extra,related_events,player_name,position_id,position_name,location,under_pressure,counterpress
0,7585,de3be98d-e227-475b-bd55-f57a6a89d308,1,769,NaN,35,Starting XI,1,1900-01-01 00:00:00.000,0,...,Colombia,0.000,"{'tactics': {'formation': 433, 'lineup': [{'pl...",[],NaN,NaN,NaN,NaN,False,False
1,7585,f50ccda4-b768-4f07-9136-8f79fd17dac5,1,768,NaN,35,Starting XI,2,1900-01-01 00:00:00.000,0,...,England,0.754,"{'tactics': {'formation': 352, 'lineup': [{'pl...",[],NaN,NaN,NaN,NaN,False,False
2,7585,b5e98805-0a22-4a5e-a306-7d40651a0f6e,1,768,NaN,18,Half Start,3,1900-01-01 00:00:00.000,0,...,England,9.320,{},[762b829f-5f24-4dd7-bfe2-da7e289838bb],NaN,NaN,NaN,NaN,False,False
3,7585,762b829f-5f24-4dd7-bfe2-da7e289838bb,1,769,NaN,18,Half Start,4,1900-01-01 00:00:00.000,0,...,Colombia,9.053,{},[b5e98805-0a22-4a5e-a306-7d40651a0f6e],NaN,NaN,NaN,NaN,False,False
4,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,769,3445.0,30,Pass,5,1900-01-01 00:00:00.240,0,...,Colombia,0.240,"{'pass': {'recipient': {'id': 5692, 'name': 'J...",[5fc9acb8-88c3-4cfb-ad9c-fc250c0dffde],Radamel Falcao García Zárate,24.0,Left Center Forward,"[60.0, 40.0]",False,False


In [4]:
home_team_id = games.iloc[0]["home_team_id"]

spadl_actions = spadl.statsbomb.convert_to_actions(
    events,
    home_team_id=home_team_id
)

spadl_actions = spadl.add_names(spadl_actions)

spadl_actions.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id,type_name,result_name,bodypart_name
0,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,0.0,769,3445.0,52.058824,34.430380,43.235294,33.569620,0,1,4,0,pass,success,foot_left
1,7585,b948f032-4c54-4782-a71a-ffeed8908d00,1,0.0,769,5692.0,43.235294,33.569620,44.117647,34.430380,21,1,0,1,dribble,success,foot
2,7585,9bdb71f9-c87b-4a66-96f0-def5312ca921,1,2.0,769,5692.0,44.117647,34.430380,40.588235,22.379747,0,1,4,2,pass,success,foot_left
3,7585,2ffa2904-8b47-4817-af26-aa9ac8d2881a,1,3.0,769,5685.0,40.588235,22.379747,42.352941,21.518987,21,1,0,3,dribble,success,foot
4,7585,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,1,4.0,769,5685.0,42.352941,21.518987,56.470588,0.860759,0,1,5,4,pass,success,foot_right


In [9]:
import socceraction.xthreat as xthreat

karun_xt_model = xthreat.load_model(
    "https://karun.in/blog/data/open_xt_12x8_v1.json"
)

In [10]:
valued_actions = spadl_actions.copy()

valued_actions["xT_added"] = karun_xt_model.rate(
    valued_actions
)

valued_actions.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id,type_name,result_name,bodypart_name,xT_added
0,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,0.0,769,3445.0,52.058824,34.430380,43.235294,33.569620,0,1,4,0,pass,success,foot_left,-0.002220
1,7585,b948f032-4c54-4782-a71a-ffeed8908d00,1,0.0,769,5692.0,43.235294,33.569620,44.117647,34.430380,21,1,0,1,dribble,success,foot,0.002220
2,7585,9bdb71f9-c87b-4a66-96f0-def5312ca921,1,2.0,769,5692.0,44.117647,34.430380,40.588235,22.379747,0,1,4,2,pass,success,foot_left,-0.002154
3,7585,2ffa2904-8b47-4817-af26-aa9ac8d2881a,1,3.0,769,5685.0,40.588235,22.379747,42.352941,21.518987,21,1,0,3,dribble,success,foot,0.000000
4,7585,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,1,4.0,769,5685.0,42.352941,21.518987,56.470588,0.860759,0,1,5,4,pass,success,foot_right,0.002044


In [7]:
player_lookup = events[
    ["player_id", "player_name"]
].dropna().drop_duplicates()

valued_actions = valued_actions.merge(
    player_lookup,
    on="player_id",
    how="left"
)

valued_actions[
    ["player_id", "player_name"]
].head()

,player_id,player_name
0,3445.0,Radamel Falcao García Zárate
1,5692.0,Juan Fernando Quintero Paniagua
2,5692.0,Juan Fernando Quintero Paniagua
3,5685.0,Carlos Alberto Sánchez Moreno
4,5685.0,Carlos Alberto Sánchez Moreno


In [8]:
player_xt = player_xt_table(
    valued_actions,
    player_col="player_name"
)

player_xt.head(20)

,player_name,total_xT,actions,avg_xT
20,Juan Guillermo Cuadrado Bello,3.180,133,0.023910
15,Johan Andrés Mojica Palacio,3.087,130,0.023746
19,Juan Fernando Quintero Paniagua,2.724,89,0.030607
8,Davinson Sánchez Mina,1.135,136,0.008346
0,Andrés Mateus Uribe Villa,0.926,46,0.020130
21,Kieran Trippier,0.799,98,0.008153
7,David Ospina Ramírez,0.787,30,0.026233
4,Carlos Arturo Bacca Ahumada,0.665,34,0.019559
28,Wílmar Enrique Barrios Terán,0.632,96,0.006583
13,Jefferson Andrés Lerma Solís,0.614,54,0.011370
